In [ ]:
%autosave 60

Autosaving every 60 seconds


## Загрузка баблиотек

In [ ]:
# !pip install razdel

In [ ]:
# !pip install wordcloud

In [ ]:
# !pip install pymorphy3

In [ ]:
# --- Системные и общие ---
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

# --- SKlearn ---
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# --- NLP-инструменты ---
import nltk
from nltk.corpus import stopwords
from razdel import tokenize as razdel_tokenize
import pymorphy3
from wordcloud import WordCloud

# --- PyTorch & Transformers ---
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification

# --- Константы и настройка среды ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Определяем устройство: MPS (Apple Silicon GPU), CUDA или CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.manual_seed(RANDOM_STATE)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(RANDOM_STATE)
else:
    device = torch.device("cpu")

print(f"✅ Используемое устройство: {device}")

# --- Инициализация лидерборда ---
leaderboard = pd.DataFrame(columns=['Метод', 'R2_Score'])

✅ Используемое устройство: cuda


## Загрузка тестовой и обучающей выборки

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PATH='/content/drive/My Drive/teta_nn_1'

In [ ]:
train_file_path = '/content/drive/My Drive/train_nn_1.csv'  # Replace with your actual file path
test_file_path = '/content/drive/My Drive/test_nn_1.csv'    # Replace with your actual file path

full_train_df = pd.read_csv(train_file_path)
test_df = pd.read_csv(test_file_path)

## Предобработка данных.
Идея - 4 колонки:
 - Локация и компания (потому что в одной и той же компании но в разных регионах платят по разному): *complocation*
 - Должность и опыт: *title*
 - Умения: *skills*
 - Описание: *description*

In [ ]:
full_train_df['experience_from']=full_train_df['experience_from'].astype(int)
test_df['experience_from']=test_df['experience_from'].astype(int)

In [ ]:
def format_experience(years):
  # Функция для прибавления годов опыта
  if years == 1:
    return f"c опытом {years} год"
  elif years >= 2 and years <= 4:
    return f"c опытом {years} года"
  elif years >= 5 and years <= 10:
    return f"c опытом {years} лет"
  else:
    return f"c опытом {years} лет" # Default for numbers outside 1-10, or you could handle this differently


In [ ]:
full_train_df['title'] = full_train_df['title'] + ' ' + full_train_df['experience_from'].apply(format_experience)
test_df['title'] = test_df['title'] + ' ' + test_df['experience_from'].apply(format_experience)

In [ ]:
# заменим пропуски пустой строкой
full_train_df = full_train_df.fillna('')
test_df = test_df.fillna('')

In [ ]:
full_train_df['complocation'] = full_train_df['location'] + ' локация ' + full_train_df['company']
test_df['complocation'] = test_df['location'] + ' локация ' + test_df['company']

In [ ]:
# Разбиваем исходный train на обучающую и валидационную выборки
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.25,
    random_state=RANDOM_STATE
)

# Очистка, лемманизация и токенизация

In [ ]:
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('russian'))
stop_words.update(['наш', 'компания', 'команда', 'работа', 'вакансия', 'искать'])


morph = pymorphy3.MorphAnalyzer()

text_cols = ['title', 'complocation', 'skills', 'description']

def preprocess_text(text: str) -> str:
    """Полный цикл предобработки текста с использованием pymorphy3."""
    # 1. Токенизация
    tokens = [token.text for token in razdel_tokenize(text)]
    # 2. Приведение к нижнему регистру и удаление не-буквенных токенов
    lower_tokens = [token.lower() for token in tokens if token.isalpha()]
    # 3. Лемматизация и удаление стоп-слов
    lemmas = [morph.parse(token)[0].normal_form for token in lower_tokens]
    cleaned_lemmas = [lemma for lemma in lemmas if lemma not in stop_words]
    return " ".join(cleaned_lemmas)


# Применяем ко всем данным
tqdm.pandas(desc="Preprocessing Train")
for col in text_cols:
  train_df[col] = train_df[col].progress_apply(preprocess_text)
tqdm.pandas(desc="Preprocessing Val")
for col in text_cols:
  val_df[col] = val_df[col].progress_apply(preprocess_text)

Preprocessing Train:   0%|          | 0/12500 [00:00<?, ?it/s]

Preprocessing Train:   0%|          | 0/12500 [00:00<?, ?it/s]

Preprocessing Train:   0%|          | 0/12500 [00:00<?, ?it/s]

Preprocessing Train:   0%|          | 0/12500 [00:00<?, ?it/s]

Preprocessing Val:   0%|          | 0/4167 [00:00<?, ?it/s]

Preprocessing Val:   0%|          | 0/4167 [00:00<?, ?it/s]

Preprocessing Val:   0%|          | 0/4167 [00:00<?, ?it/s]

Preprocessing Val:   0%|          | 0/4167 [00:00<?, ?it/s]

In [ ]:
# Создаю словарь
corpus = []
for col in text_cols:
    for text in train_df[col]:
        corpus.extend(text.split())

# Строю словарь
word_counts = Counter(corpus)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)
word_to_idx = {word: i + 2 for i, word in enumerate(vocab)}
word_to_idx['<PAD>'] = 0
word_to_idx['<UNK>'] = 1
vocab_size = len(word_to_idx)



## Построение архитектуры и обучение модели
Идея — сделать четыре BiLSTM башни, каждая учит свой текстовый столбец (описаны выше). Далее сделать склейку и прогнать объединённую последовательность через два шага внимания.


In [ ]:
# Dataset
class VacancyDataset(Dataset):
    def __init__(self, df, word_to_idx, text_columns, max_len=100, train_flag=True):
        self.df = df
        self.word_to_idx = word_to_idx
        self.text_columns = text_columns
        self.max_len = max_len
        self.train_flag=train_flag
        if train_flag:
          self.scores = df['log_salary_from'].values  # целевая переменная

    def __len__(self):
        return len(self.df)

    def _encode_text(self, text):
        tokens = text.split()
        token_ids = [self.word_to_idx.get(token, self.word_to_idx['<UNK>']) for token in tokens]
        if len(token_ids) > self.max_len:
            token_ids = token_ids[:self.max_len]
        else:
            token_ids += [self.word_to_idx['<PAD>']] * (self.max_len - len(token_ids))
        return torch.tensor(token_ids, dtype=torch.long)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        encoded_texts = [self._encode_text(row[col]) for col in self.text_columns]
        if self.train_flag:
          target = torch.tensor(self.scores[idx], dtype=torch.float32)
          return encoded_texts, target
        else:
          return encoded_texts



In [ ]:
train_dataset = VacancyDataset(train_df, word_to_idx, text_columns=text_cols)
val_dataset = VacancyDataset(val_df, word_to_idx, text_columns=text_cols)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
# Загружаю тренировочную и валидационную выборку
train_dataset = VacancyDataset(train_df, word_to_idx, text_cols)
val_dataset = VacancyDataset(val_df, word_to_idx, text_cols)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# Архитектура модели
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1, num_columns=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_columns = num_columns
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding_dropout = nn.Dropout(0.2)

        self.lstms = nn.ModuleList([
            nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                    bidirectional=True, batch_first=True)
            for _ in range(num_columns)
        ])



        self.attn = nn.Sequential(
          nn.Linear(hidden_dim*2, 128),
          nn.Tanh(),
          nn.Linear(128, 1)
        )

        self.query_proj = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.key_proj   = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.value_proj = nn.Linear(hidden_dim * 2, hidden_dim * 2)


        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim*2*2, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x_list):
        batch_size = x_list[0].size(0)
        device = x_list[0].device

        all_lstm_outs = []
        all_masks = []

        for i in range(self.num_columns):
            x = x_list[i]
            mask = (x != 0).unsqueeze(-1)

            embedded = self.embedding_dropout(self.embedding(x))
            lstm_out, _ = self.lstms[i](embedded)

            all_lstm_outs.append(lstm_out)
            all_masks.append(mask)

        merged_seq = torch.cat(all_lstm_outs, dim=1)
        merged_mask = torch.cat(all_masks, dim=1)

        # Attention
        attn_logits = self.attn(merged_seq)
        attn_logits = attn_logits.masked_fill(~merged_mask, -1e9)
        attn_weights = torch.softmax(attn_logits, dim=1)

        context = torch.sum(attn_weights * merged_seq, dim=1)

        Q = self.query_proj(merged_seq)
        K = self.key_proj(merged_seq)
        V = self.value_proj(merged_seq)


        scores = torch.bmm(Q, K.transpose(1, 2)) / (self.hidden_dim ** 0.5)  # (B, L, L)
        scores = scores.masked_fill(~merged_mask.squeeze(-1).unsqueeze(1), -1e9)
        attn_matrix = torch.softmax(scores, dim=-1)

        cross_attn_output = torch.bmm(attn_matrix, V)
        cross_context = cross_attn_output.mean(dim=1)

        # Финальный регрессор
        combined = torch.cat([context, cross_context], dim=1)
        out = self.regressor(combined)
        return out.squeeze(1)

# Обучение
model_lstm = BiLSTMRegressor(vocab_size=len(word_to_idx), embedding_dim=128, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=0.001)
criterion = nn.MSELoss()
NUM_EPOCHS = 40

for epoch in range(NUM_EPOCHS):
    model_lstm.train()
    total_loss = 0
    for texts, scores in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        texts = [t.to(device) for t in texts]
        scores = scores.to(device)
        optimizer.zero_grad()
        outputs = model_lstm(texts)
        loss = criterion(outputs, scores)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Train Loss: {total_loss / len(train_loader):.4f}")

# Оценка модели
model_lstm.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for texts, scores in val_loader:
        texts = [t.to(device) for t in texts]
        scores = scores.to(device)
        outputs = model_lstm(texts)
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(scores.cpu().numpy())

r2_lstm = r2_score(all_labels, all_preds)
print(f"R² score для Bi-LSTM: {r2_lstm:.4f}")

Epoch 1/40:   0%|          | 0/391 [00:00<?, ?it/s]

Train Loss: 0.7937


Epoch 2/40:   0%|          | 0/391 [00:00<?, ?it/s]

Train Loss: 0.2461


Epoch 3/40:   0%|          | 0/391 [00:00<?, ?it/s]

...

Train Loss: 0.0401


Epoch 39/40:   0%|          | 0/391 [00:00<?, ?it/s]

Train Loss: 0.0382


Epoch 40/40:   0%|          | 0/391 [00:00<?, ?it/s]

Train Loss: 0.0370
R² score для Bi-LSTM: 0.6592


## Формирование сабмита

In [ ]:
tqdm.pandas(desc="Preprocessing Test")
for col in text_cols:
  test_df[col] = test_df[col].progress_apply(preprocess_text)


Preprocessing Test:   0%|          | 0/5556 [00:00<?, ?it/s]

Preprocessing Test:   0%|          | 0/5556 [00:00<?, ?it/s]

Preprocessing Test:   0%|          | 0/5556 [00:00<?, ?it/s]

Preprocessing Test:   0%|          | 0/5556 [00:00<?, ?it/s]

In [ ]:
test_dataset=VacancyDataset(test_df, word_to_idx, text_cols, train_flag=False)
test_dataloader = DataLoader(test_dataset, shuffle=False)

In [ ]:
pred_log = []
with torch.no_grad():
  for x in tqdm(test_dataloader, desc="Predict"):
    texts = [t.to(device) for t in x]
    out = model_lstm(texts)
    pred_log.extend(out.cpu().numpy())

Predict:   0%|          | 0/5556 [00:00<?, ?it/s]

In [ ]:
sap = pd.read_csv('sample_submition.csv', index_col='index')

In [ ]:
sap['prediction']=pred_log

In [ ]:
sap.to_csv('submition.csv')